# Notebook 20 — Pipeline SAM2 para detección de armas

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Evaluar si la incorporación de SAM2 como etapa de segmentación de objetos mejora la detección de armas respecto al baseline (Config A — frame completo).

## Pipeline propuesto (Config SAM2)

```
frame
  → [Stage 1] yolov8s-seg     → bbox persona + padding C10
  → [Stage 2] SAM2ImagePredictor (prompt=bbox persona)
                               → N máscaras de objetos en la región
  → [Stage 3] filtro tamaño   → descartar segmentos pequeños (ruido)
  → [Stage 4] weapon_model    → clasificar cada segmento como arma / no arma
  → si algún segmento es arma → frame positivo
  → acumular frames           → umbral 5 → clasificación clip
```

## Comparativa final

| Config | Descripción |
|--------|-------------|
| **A** | Frame completo (baseline) |
| **C10** | BBox persona +10% padding, sin SAM2 |
| **SAM2** | BBox persona +10% padding + SAM2 + weapon_model por segmento |

**Dataset:** GAR test set — 140 clips positivos + 118 clips negativos  
**Modelo armas:** `yolov8m_weapons_B_e50_640` · CONF=0.25  
**SAM2:** `facebook/sam2.1-hiera-base-plus` (balance velocidad/precisión en T4)

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sam2 ultralytics opencv-python
print('✅ Dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 13.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.1 MB/s eta 0:00:00
✅ Dependencias instaladas


In [ ]:
import os
import json
import shutil
import csv
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import torch
from ultralytics import YOLO
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ── CONFIG ────────────────────────────────────────────────────────────────────
POS_LIST   = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST   = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR    = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/sam2_ablation'

WEAPON_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'

IMG_SIZE   = 640
CONF_SEG   = 0.25       # confianza yolov8s-seg para personas
CONF_WEAPON = 0.25     # confianza weapon_model
IOU_NMS    = 0.7
PADDING    = 0.10      # padding C10 sobre bbox de persona
MAX_SECONDS = 15       # None para clip completo
DETECTION_THRESHOLD = 5  # frames mínimos con arma para clip positivo

# Parámetros SAM2
SAM2_MODEL_ID  = 'facebook/sam2.1-hiera-base-plus'  # balance velocidad/precisión en T4
MIN_MASK_AREA_RATIO = 0.02   # segmento debe ser >= 2% del área del bbox de persona
MAX_MASKS_PER_FRAME = 5      # top-N segmentos por área (descartar ruido pequeño)
# ─────────────────────────────────────────────────────────────────────────────

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

CAT_LABELS = {
    'N1': 'Walking empty hands',  'N2': 'Jogging',
    'N3': 'Running',              'N4': 'Sneaking empty hands',
    'N5': 'Phone relaxed',        'N6': 'Phone looking',
    'N7': 'Phone both hands',     'N8': 'Phone recording 1h',
    'N9': 'Phone recording 2h',   'N10': 'Water bottle relaxed',
    'N11': 'Drinking',            'N12': 'Holding heavy object',
}

print('✅ Config cargada')
print(f'   SAM2 model : {SAM2_MODEL_ID}')
print(f'   PADDING    : {PADDING*100:.0f}%')
print(f'   MIN_MASK   : {MIN_MASK_AREA_RATIO*100:.0f}% del bbox persona')
print(f'   MAX_MASKS  : {MAX_MASKS_PER_FRAME} por frame')

✅ Config cargada
   SAM2 model : facebook/sam2.1-hiera-base-plus
   PADDING    : 10%
   MIN_MASK   : 2% del bbox persona
   MAX_MASKS  : 5 por frame


---
## 1. Carga de modelos

In [ ]:
# Copiar weights a /content/ para evitar desconexiones de Drive durante inferencia
!cp '{WEAPON_WEIGHTS}' /content/weapon_best.pt

# Modelo de segmentación de personas
seg_model = YOLO('yolov8s-seg.pt')
print('✅ yolov8s-seg cargado')

# Modelo de detección de armas
weapon_model = YOLO('/content/weapon_best.pt')
print('✅ weapon_model (Modelo B) cargado')

# SAM2 — descarga automática desde HuggingFace (~180MB base-plus)
sam2_predictor = SAM2ImagePredictor.from_pretrained(SAM2_MODEL_ID)
print(f'✅ SAM2 ({SAM2_MODEL_ID}) cargado')

✅ yolov8s-seg cargado
✅ weapon_model (Modelo B) cargado


✅ SAM2 (facebook/sam2.1-hiera-base-plus) cargado


---
## 2. Funciones auxiliares

In [ ]:
def get_person_bbox_padded(frame, padding=PADDING):
    """
    Detecta personas con yolov8s-seg y devuelve el bbox expandido
    con padding de la persona de mayor área.
    Devuelve (x1, y1, x2, y2) en píxeles, o None si no hay persona.
    """
    h, w = frame.shape[:2]
    results = seg_model.predict(
        frame, imgsz=IMG_SIZE, conf=CONF_SEG,
        classes=[0], verbose=False, device='cuda'
    )[0]

    if results.boxes is None or len(results.boxes) == 0:
        return None

    # Seleccionar persona de mayor área de bbox
    best_box = None
    best_area = 0
    for box in results.boxes:
        x1, y1, x2, y2 = map(float, box.xyxy[0].cpu())
        area = (x2 - x1) * (y2 - y1)
        if area > best_area:
            best_area = area
            best_box = (x1, y1, x2, y2)

    if best_box is None:
        return None

    x1, y1, x2, y2 = best_box
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - bw * padding))
    y1 = max(0, int(y1 - bh * padding))
    x2 = min(w, int(x2 + bw * padding))
    y2 = min(h, int(y2 + bh * padding))
    return (x1, y1, x2, y2)


def get_sam2_masks(frame_rgb, bbox):
    """
    Usa SAM2ImagePredictor con el bbox de la persona como prompt.
    Devuelve lista de máscaras binarias (H, W) ordenadas por área descendente,
    filtradas por MIN_MASK_AREA_RATIO y limitadas a MAX_MASKS_PER_FRAME.
    """
    x1, y1, x2, y2 = bbox
    bbox_area = (x2 - x1) * (y2 - y1)
    min_area = bbox_area * MIN_MASK_AREA_RATIO

    # SAM2 espera RGB
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        sam2_predictor.set_image(frame_rgb)
        # Prompt: bbox de la persona → SAM2 segmenta objetos dentro
        input_box = np.array([[x1, y1, x2, y2]], dtype=np.float32)
        masks, scores, _ = sam2_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=input_box,
            multimask_output=True,  # devuelve 3 máscaras candidatas por prompt
        )

    # masks shape: (N, H, W) — ordenar por área y filtrar
    mask_list = []
    for mask in masks:
        area = mask.sum()
        if area >= min_area:
            mask_list.append((area, mask.astype(np.uint8)))

    # Ordenar por área descendente y limitar
    mask_list.sort(key=lambda x: x[0], reverse=True)
    return [m for _, m in mask_list[:MAX_MASKS_PER_FRAME]]


def apply_mask_to_frame(frame, mask):
    """
    Aplica una máscara binaria al frame — fondo negro, objeto visible.
    Devuelve el frame enmascarado.
    """
    masked = np.zeros_like(frame)
    masked[mask == 1] = frame[mask == 1]
    return masked


def detect_weapon_in_masks(frame, masks):
    """
    Para cada máscara, aplica al frame y pasa al weapon_model.
    Devuelve True si algún segmento supera CONF_WEAPON.
    También devuelve las predicciones para mAP (sobre frame completo si hay detección).
    """
    for mask in masks:
        masked_frame = apply_mask_to_frame(frame, mask)
        result = weapon_model.predict(
            masked_frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
            iou=IOU_NMS, verbose=False, device='cuda'
        )[0]
        if result.boxes is not None and len(result.boxes) > 0:
            preds = []
            for b in result.boxes:
                x1, y1, x2, y2 = map(float, b.xyxy[0])
                conf = float(b.conf[0])
                preds.append((x1, y1, x2, y2, conf))
            return True, preds
    return False, []


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)


def load_gt_boxes(label_json_path):
    with open(label_json_path) as f:
        data = json.load(f)
    gt = defaultdict(list)
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        gt[ann['image_id']].append((x, y, x+w, y+h))
    return gt


print('✅ Funciones auxiliares definidas')

✅ Funciones auxiliares definidas


---
## 3. Función principal de procesado de vídeo

In [ ]:
def process_video_sam2(video_path, gt_boxes=None):
    """
    Procesa un vídeo con el pipeline SAM2.
    Devuelve:
        - frames_with_gun: nº de frames con arma detectada
        - total_frames: nº total de frames procesados
        - frame_preds: lista de (frame_idx, preds) para mAP si gt_boxes no es None
        - no_person_frames: nº de frames sin persona detectada
        - no_mask_frames: nº de frames con persona pero sin máscaras SAM2
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = n_frames
    if MAX_SECONDS is not None and fps and fps > 0:
        max_frames = min(max_frames, int(MAX_SECONDS * fps))

    frames_with_gun  = 0
    no_person_frames = 0
    no_mask_frames   = 0
    frame_preds      = []  # para mAP

    frame_i = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok or frame_i >= max_frames:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # Stage 1: detectar persona y obtener bbox con padding
        bbox = get_person_bbox_padded(frame_bgr)

        if bbox is None:
            # Sin persona → fallback a frame completo
            no_person_frames += 1
            result = weapon_model.predict(
                frame_bgr, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                iou=IOU_NMS, verbose=False, device='cuda'
            )[0]
            if result.boxes is not None and len(result.boxes) > 0:
                frames_with_gun += 1
                if gt_boxes is not None:
                    preds = [(float(b.xyxy[0][0]), float(b.xyxy[0][1]),
                              float(b.xyxy[0][2]), float(b.xyxy[0][3]),
                              float(b.conf[0])) for b in result.boxes]
                    frame_preds.append((frame_i + 1, preds))
            frame_i += 1
            continue

        # Stage 2: SAM2 con bbox como prompt
        masks = get_sam2_masks(frame_rgb, bbox)

        if len(masks) == 0:
            # Sin máscaras → fallback a bbox recortado
            no_mask_frames += 1
            x1, y1, x2, y2 = bbox
            crop = frame_bgr[y1:y2, x1:x2]
            result = weapon_model.predict(
                crop, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                iou=IOU_NMS, verbose=False, device='cuda'
            )[0]
            if result.boxes is not None and len(result.boxes) > 0:
                frames_with_gun += 1
            frame_i += 1
            continue

        # Stage 3+4: clasificar cada segmento con weapon_model
        weapon_found, preds = detect_weapon_in_masks(frame_bgr, masks)
        if weapon_found:
            frames_with_gun += 1
            if gt_boxes is not None:
                frame_preds.append((frame_i + 1, preds))

        frame_i += 1

    cap.release()
    return frames_with_gun, max_frames, frame_preds, no_person_frames, no_mask_frames


print('✅ process_video_sam2() definida')

✅ process_video_sam2() definida


---
## 4. Validación en subconjunto pequeño (20 clips)

Antes de lanzar la evaluación completa, validamos el pipeline sobre:
- 10 clips positivos (variedad de cámaras y posiciones)
- 10 clips negativos de las categorías más problemáticas (N8, N9, N6)

Objetivo: verificar que el pipeline funciona end-to-end y estimar el tiempo por clip.

In [ ]:
import time

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]
neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]

# Seleccionar subconjunto representativo
# 10 positivos: primeros 5 PAH + primeros 5 PCH
pah = [p for p in pos_paths if 'PAH' in p.parent.name][:5]
pch = [p for p in pos_paths if 'PCH' in p.parent.name][:5]
val_pos = pah + pch

# 10 negativos: 4 N8 + 3 N9 + 3 N6
n8 = [p for p in neg_paths if p.parent.name.startswith('N8')][:4]
n9 = [p for p in neg_paths if p.parent.name.startswith('N9')][:3]
n6 = [p for p in neg_paths if p.parent.name.startswith('N6')][:3]
val_neg = n8 + n9 + n6

print(f'Validación sobre {len(val_pos)} clips positivos y {len(val_neg)} clips negativos')
print()

val_results = []
times = []

for vp, true_label in [(p, 1) for p in val_pos] + [(p, 0) for p in val_neg]:
    if not vp.exists():
        print(f'  ❌ No existe: {vp}')
        continue

    clip_id = vp.parent.name
    local_in = '/content/tmp_val.mp4'
    shutil.copy2(str(vp), local_in)

    t0 = time.time()
    n_gun, n_total, _, n_noperson, n_nomask = process_video_sam2(local_in)
    elapsed = time.time() - t0
    times.append(elapsed)

    os.remove(local_in)

    pred = 1 if n_gun >= DETECTION_THRESHOLD else 0
    correct = pred == true_label
    val_results.append({'clip': clip_id, 'true': true_label, 'pred': pred, 'correct': correct})

    label_str = 'TP' if (pred==1 and true_label==1) else \
                'TN' if (pred==0 and true_label==0) else \
                'FP' if (pred==1 and true_label==0) else 'FN'
    icon = '✅' if correct else '❌'
    print(f'  {icon} {clip_id}: {n_gun}/{n_total} frames con arma → {label_str} '
          f'[{elapsed:.1f}s | sin persona:{n_noperson} | sin máscara:{n_nomask}]')

n_correct = sum(r['correct'] for r in val_results)
print(f'\n  Accuracy validación : {n_correct}/{len(val_results)} ({n_correct/len(val_results)*100:.1f}%)')
print(f'  Tiempo medio/clip   : {np.mean(times):.1f}s')
print(f'  Tiempo total estimado (258 clips): {np.mean(times)*258/3600:.1f}h')

Validación sobre 10 clips positivos y 10 clips negativos

  ❌ PAH1_C1_P1_V1_HB_3: 3/150 frames con arma → FN [58.1s | sin persona:44 | sin máscara:0]
  ✅ PAH1_C1_P1_V1_HB_4: 29/175 frames con arma → TP [66.4s | sin persona:49 | sin máscara:0]
  ✅ PAH1_C1_P2_V1_HB_1: 78/275 frames con arma → TP [79.7s | sin persona:127 | sin máscara:0]
  ✅ PAH1_C1_P2_V1_HB_3: 31/225 frames con arma → TP [75.2s | sin persona:85 | sin máscara:0]
  ✅ PAH1_C1_P4_V1_HB_1: 13/150 frames con arma → TP [51.7s | sin persona:55 | sin máscara:0]
  ❌ PCH1_C1_P1_V1_HB_1: 0/150 frames con arma → FN [58.2s | sin persona:41 | sin máscara:0]
  ✅ PCH1_C1_P2_V1_HB_2: 6/225 frames con arma → TP [63.5s | sin persona:110 | sin máscara:0]
  ✅ PCH1_C1_P2_V1_HB_4: 9/225 frames con arma → TP [75.2s | sin persona:86 | sin máscara:0]
  ✅ PCH1_C1_P4_V1_HB_2: 5/125 frames con arma → TP [53.1s | sin persona:25 | sin máscara:0]
  ❌ PCH1_C1_P4_V1_HB_4: 0/150 frames con arma → FN [50.5s | sin persona:57 | sin máscara:0]
  ✅ N8_C1_P1_V1_

---
## 5. Evaluación completa — BLOQUE A: mAP a nivel de frame

In [ ]:
IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)
all_tp = defaultdict(list)
all_fp = defaultdict(list)
total_gt = 0

print('\n' + '='*60)
print('BLOQUE A — mAP a nivel de frame')
print('='*60)

for vp in pos_paths:
    if not vp.exists():
        continue

    label_path = vp.parent / 'label.json'
    if not label_path.exists():
        continue

    clip_id  = vp.parent.name
    local_in = f'/content/{clip_id}_eval.mp4'
    shutil.copy2(str(vp), local_in)

    gt_boxes = load_gt_boxes(label_path)
    total_gt += sum(len(v) for v in gt_boxes.values())

    _, _, frame_preds, _, _ = process_video_sam2(local_in, gt_boxes=gt_boxes)

    for (frame_idx, preds) in frame_preds:
        gts = gt_boxes.get(frame_idx, [])
        for iou_thr in IOU_THRESHOLDS:
            matched_gt = set()
            for (px1, py1, px2, py2, conf) in sorted(preds, key=lambda x: -x[4]):
                best_iou, best_j = 0, -1
                for j, gt in enumerate(gts):
                    if j in matched_gt: continue
                    s = iou((px1, py1, px2, py2), gt)
                    if s > best_iou:
                        best_iou, best_j = s, j
                if best_iou >= iou_thr and best_j >= 0:
                    all_tp[iou_thr].append(1); all_fp[iou_thr].append(0)
                    matched_gt.add(best_j)
                else:
                    all_tp[iou_thr].append(0); all_fp[iou_thr].append(1)

    os.remove(local_in)
    print(f'  ✅ {clip_id}')

aps = {}
for iou_thr in IOU_THRESHOLDS:
    tp = sum(all_tp[iou_thr])
    fp = sum(all_fp[iou_thr])
    fn = total_gt - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    aps[iou_thr] = {'precision': precision, 'recall': recall}

map50   = aps[0.5]['precision']
map5095 = np.mean([v['precision'] for v in aps.values()])

print(f'\n  Total GT boxes  : {total_gt}')
print(f'  mAP@50          : {map50:.4f}')
print(f'  mAP@50:95       : {map5095:.4f}')
print(f'  Precision@50    : {aps[0.5]["precision"]:.4f}')
print(f'  Recall@50       : {aps[0.5]["recall"]:.4f}')


BLOQUE A — mAP a nivel de frame
  ✅ PAH1_C1_P1_V1_HB_3
  ✅ PAH1_C1_P1_V1_HB_4
  ✅ PAH1_C1_P2_V1_HB_1
  ✅ PAH1_C1_P2_V1_HB_3
  ✅ PAH1_C1_P4_V1_HB_1
  ✅ PAH1_C1_P4_V1_HB_2
  ✅ PAH1_C2_P3_V1_HB_1
  ✅ PAH1_C2_P3_V1_HB_3
  ✅ PAH1_C2_P3_V2_HB_2
  ✅ PAH1_C2_P3_V2_HB_3
  ✅ PAH1_C2_P5_V1_HB_1
  ✅ PAH1_C2_P5_V1_HB_4
  ✅ PAH1_C2_P5_V2_HB_3
  ✅ PAH1_C2_P5_V2_HB_4
  ✅ PAH2_C1_P1_V1_HB_1
  ✅ PAH2_C1_P1_V1_HB_4
  ✅ PAH2_C1_P2_V1_HB_3
  ✅ PAH2_C1_P2_V1_HB_4
  ✅ PAH2_C1_P4_V1_HB_1
  ✅ PAH2_C1_P4_V1_HB_3
  ✅ PAH2_C2_P3_V1_HB_1
  ✅ PAH2_C2_P3_V1_HB_4
  ✅ PAH2_C2_P3_V2_HB_2
  ✅ PAH2_C2_P3_V2_HB_4
  ✅ PAH2_C2_P5_V1_HB_3
  ✅ PAH2_C2_P5_V1_HB_4
  ✅ PAH2_C2_P5_V2_HB_2
  ✅ PAH2_C2_P5_V2_HB_4
  ✅ PAH3_C1_P1_V1_HB_1
  ✅ PAH3_C1_P1_V1_HB_4
  ✅ PAH3_C1_P2_V1_HB_1
  ✅ PAH3_C1_P2_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_4
  ✅ PAH3_C2_P3_V1_HB_1
  ✅ PAH3_C2_P3_V1_HB_2
  ✅ PAH3_C2_P3_V2_HB_1
  ✅ PAH3_C2_P3_V2_HB_3
  ✅ PAH3_C2_P5_V1_HB_2
  ✅ PAH3_C2_P5_V1_HB_4
  ✅ PAH3_C2_P5_V2_HB_2
  ✅ PAH3_C2_P5_V2_HB_4
 

---
## 6. Evaluación completa — BLOQUE B: Clasificación a nivel de clip

In [ ]:
print('\n' + '='*60)
print(f'BLOQUE B — Clasificación a nivel de clip (umbral={DETECTION_THRESHOLD} frames)')
print('='*60)

y_true, y_pred = [], []
clip_details   = []

def classify_clip_sam2(video_path, true_label):
    local_in = '/content/tmp_cls.mp4'
    shutil.copy2(str(video_path), local_in)
    n_gun, n_total, _, n_noperson, n_nomask = process_video_sam2(local_in)
    os.remove(local_in)
    pred = 1 if n_gun >= DETECTION_THRESHOLD else 0
    return pred, n_gun, n_total, n_noperson, n_nomask

# Positivos
print('\n  Procesando positivos...')
for vp in pos_paths:
    if not vp.exists(): continue
    clip_id = vp.parent.name
    pred, n_det, n_total, n_nop, n_nom = classify_clip_sam2(vp, 1)
    y_true.append(1); y_pred.append(pred)
    clip_details.append({'clip': clip_id, 'true': 1, 'pred': pred,
                         'det_frames': n_det, 'total_frames': n_total,
                         'no_person': n_nop, 'no_mask': n_nom, 'category': 'POS'})
    print(f'    {clip_id}: {n_det}/{n_total} frames → {"✅TP" if pred==1 else "❌FN"} '
          f'[sin persona:{n_nop} | sin máscara:{n_nom}]')

# Negativos
print(f'\n  Procesando negativos ({len(neg_paths)} clips)...')
for vp in neg_paths:
    if not vp.exists(): continue
    clip_id  = vp.parent.name
    category = clip_id.split('_')[0]
    pred, n_det, n_total, n_nop, n_nom = classify_clip_sam2(vp, 0)
    y_true.append(0); y_pred.append(pred)
    clip_details.append({'clip': clip_id, 'true': 0, 'pred': pred,
                         'det_frames': n_det, 'total_frames': n_total,
                         'no_person': n_nop, 'no_mask': n_nom, 'category': category})
    print(f'    {clip_id}: {n_det}/{n_total} frames → {"❌FP" if pred==1 else "✅TN"}')

# Métricas globales
y_true = np.array(y_true)
y_pred = np.array(y_pred)

TP = int(((y_true==1) & (y_pred==1)).sum())
TN = int(((y_true==0) & (y_pred==0)).sum())
FP = int(((y_true==0) & (y_pred==1)).sum())
FN = int(((y_true==1) & (y_pred==0)).sum())

accuracy  = (TP + TN) / len(y_true)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\n  --- Métricas globales (clip-level) ---')
print(f'  Clips evaluados : {len(y_true)} ({TP+FN} pos / {TN+FP} neg)')
print(f'  Accuracy        : {accuracy:.4f}')
print(f'  Precision       : {precision:.4f}')
print(f'  Recall          : {recall:.4f}')
print(f'  F1              : {f1:.4f}')
print(f'\n  Matriz de confusión:')
print(f'                Pred POS   Pred NEG')
print(f'  Real POS    |   {TP:4d}   |   {FN:4d}  |')
print(f'  Real NEG    |   {FP:4d}   |   {TN:4d}  |')


BLOQUE B — Clasificación a nivel de clip (umbral=5 frames)

  Procesando positivos...
    PAH1_C1_P1_V1_HB_3: 3/150 frames → ❌FN [sin persona:44 | sin máscara:0]
    PAH1_C1_P1_V1_HB_4: 29/175 frames → ✅TP [sin persona:49 | sin máscara:0]
    PAH1_C1_P2_V1_HB_1: 78/275 frames → ✅TP [sin persona:127 | sin máscara:0]
    PAH1_C1_P2_V1_HB_3: 31/225 frames → ✅TP [sin persona:85 | sin máscara:0]
    PAH1_C1_P4_V1_HB_1: 13/150 frames → ✅TP [sin persona:55 | sin máscara:0]
    PAH1_C1_P4_V1_HB_2: 25/150 frames → ✅TP [sin persona:55 | sin máscara:0]
    PAH1_C2_P3_V1_HB_1: 31/270 frames → ✅TP [sin persona:122 | sin máscara:0]
    PAH1_C2_P3_V1_HB_3: 15/270 frames → ✅TP [sin persona:102 | sin máscara:0]
    PAH1_C2_P3_V2_HB_2: 45/330 frames → ✅TP [sin persona:120 | sin máscara:0]
    PAH1_C2_P3_V2_HB_3: 37/300 frames → ✅TP [sin persona:108 | sin máscara:0]
    PAH1_C2_P5_V1_HB_1: 44/240 frames → ✅TP [sin persona:62 | sin máscara:0]
    PAH1_C2_P5_V1_HB_4: 97/270 frames → ✅TP [sin persona:85 | 

---
## 7. Evaluación completa — BLOQUE C: FP por categoría negativa

In [ ]:
print('\n' + '='*60)
print('BLOQUE C — Falsos positivos por categoría negativa')
print('='*60)

neg_details = [d for d in clip_details if d['true'] == 0]
cat_stats   = defaultdict(lambda: {'total': 0, 'fp': 0})

for d in neg_details:
    cat = d['category']
    cat_stats[cat]['total'] += 1
    if d['pred'] == 1:
        cat_stats[cat]['fp'] += 1

print(f"\n  {'Cat':<5} {'Descripción':<28} {'FP':>4} {'Total':>6} {'FP%':>7}")
print('  ' + '-'*55)
for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
    s = cat_stats[cat]
    fp_rate = s['fp'] / s['total'] * 100 if s['total'] > 0 else 0
    desc = CAT_LABELS.get(cat, '')
    print(f"  {cat:<5} {desc:<28} {s['fp']:>4} {s['total']:>6} {fp_rate:>6.1f}%")


BLOQUE C — Falsos positivos por categoría negativa

  Cat   Descripción                    FP  Total     FP%
  -------------------------------------------------------
  N1    Walking empty hands             6      9   66.7%
  N2    Jogging                         6      9   66.7%
  N3    Running                         5      8   62.5%
  N4    Sneaking empty hands            3      7   42.9%
  N5    Phone relaxed                   5     10   50.0%
  N6    Phone looking                  11     16   68.8%
  N7    Phone both hands                4      7   57.1%
  N8    Phone recording 1h             10     15   66.7%
  N9    Phone recording 2h              6      9   66.7%
  N10   Water bottle relaxed            7      9   77.8%
  N11   Drinking                        6      9   66.7%
  N12   Holding heavy object            7     10   70.0%


---
## 8. Guardar resultados y tabla comparativa final

In [ ]:
# ── Guardar CSV de clips
csv_path = Path(OUT_DIR) / 'clip_results_SAM2.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['clip','true','pred','det_frames',
                                           'total_frames','no_person','no_mask','category'])
    writer.writeheader()
    writer.writerows(clip_details)

# ── Guardar TXT de resultados
txt_path = Path(OUT_DIR) / 'results_config_SAM2.txt'
with open(txt_path, 'w') as f:
    f.write('=== EVALUACIÓN SAM2 PIPELINE ===\n\n')
    f.write(f'SAM2 model     : {SAM2_MODEL_ID}\n')
    f.write(f'CONF_WEAPON={CONF_WEAPON} | PADDING={PADDING} | THRESHOLD={DETECTION_THRESHOLD}\n')
    f.write(f'MIN_MASK_AREA={MIN_MASK_AREA_RATIO} | MAX_MASKS={MAX_MASKS_PER_FRAME}\n\n')
    f.write(f'--- BLOQUE A ---\n')
    f.write(f'mAP@50    : {map50:.4f}\n')
    f.write(f'mAP@50:95 : {map5095:.4f}\n\n')
    f.write(f'--- BLOQUE B ---\n')
    f.write(f'Accuracy  : {accuracy:.4f}\n')
    f.write(f'Precision : {precision:.4f}\n')
    f.write(f'Recall    : {recall:.4f}\n')
    f.write(f'F1        : {f1:.4f}\n')
    f.write(f'TP={TP} TN={TN} FP={FP} FN={FN}\n\n')
    f.write('--- BLOQUE C ---\n')
    for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
        s = cat_stats[cat]
        fp_rate = s['fp'] / s['total'] * 100 if s['total'] > 0 else 0
        f.write(f'  {cat}: {s["fp"]}/{s["total"]} ({fp_rate:.1f}%)\n')

print(f'✅ Resultados guardados en: {OUT_DIR}')

# ── Tabla comparativa final
print('\n' + '='*60)
print('TABLA COMPARATIVA FINAL')
print('='*60)
print(f"  {'Config':<30} {'F1':>7} {'Prec':>7} {'Rec':>7} {'FP':>5} {'FN':>5}")
print('  ' + '-'*60)

# Resultados anteriores (de notebooks 10 y 17)
prev = [
    ('A — Sin seg (baseline)',     0.7949, 0.7209, 0.8857, 48, 16),
    ('C10 — BBox +10% padding',    0.7947, 0.7407, 0.8571, 42, 20),
    ('SAM2 — bbox prompt',         f1,     precision, recall, FP, FN),
]
for name, f1_, prec, rec, fp_, fn_ in prev:
    marker = ' ← NUEVO' if 'SAM2' in name else ''
    print(f"  {name:<30} {f1_:>7.4f} {prec:>7.4f} {rec:>7.4f} {fp_:>5} {fn_:>5}{marker}")

# Veredicto
print()
if f1 > 0.7949:
    print('  ✅ SAM2 MEJORA el baseline (Config A)')
elif f1 > 0.7947:
    print('  ≈  SAM2 equivalente a C10, no mejora el baseline')
else:
    print('  ❌ SAM2 NO mejora el baseline — documentar como resultado negativo')

print('\n✅ Evaluación completa.')

✅ Resultados guardados en: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/sam2_ablation

TABLA COMPARATIVA FINAL
  Config                              F1    Prec     Rec    FP    FN
  ------------------------------------------------------------
  A — Sin seg (baseline)          0.7949  0.7209  0.8857    48    16
  C10 — BBox +10% padding         0.7947  0.7407  0.8571    42    20
  SAM2 — bbox prompt              0.7294  0.6200  0.8857    76    16 ← NUEVO

  ❌ SAM2 NO mejora el baseline — documentar como resultado negativo

✅ Evaluación completa.
